# M3GNet XAS pipeline with graph-cache precomputation (showcase)

Duplicate of the `train_m3gnet_xas_pipeline.py` pipeline, run end to end from this
notebook with the graph-cache precomputation described in `research/`:

1. **Stage 1** precomputes (graph, line_graph) pairs for all ~15k FEFF materials
   once, from the encoder settings the pipeline actually uses, verifies the cache
   against the on-the-fly build, and saves one file per task/split under
   `output/graph_cache/`.
2. **Stage 2** runs the full pipeline on the verified cache: scratch M3GNetXAS
   encoder (up to 1000 epochs, early stopping 60) -> 64D feature export ->
   UniversalXAS (800 epochs) -> 8 validation-selected tuned heads (1000 epochs
   each) -> validation/test evaluation.

Settings:

- seed: 45555 (encoder, heads, and balanced sampling)
- run name: `m3gnet_xas_precomputation_showcase`
- encoder: the pipeline's default M3GNetXASEncoder (64D, cutoff 4.0, threebody_cutoff 4.0)
- graph cache: `output/graph_cache/feff_graphs_c4_tb4` (one directory per setting)
- selection: validation only; test is evaluated exactly once by the pipeline


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'tutorial_omnixas' / 'train_m3gnet_xas_pipeline.py').is_file():
    REPO_ROOT = REPO_ROOT.parent

SEED = 45555
RUN_NAME = 'm3gnet_xas_precomputation_showcase'
OUTPUT_ROOT = REPO_ROOT / 'output' / 'training' / 'm3gnet_xas_pipeline'
PIPELINE = REPO_ROOT / 'tutorial_omnixas' / 'train_m3gnet_xas_pipeline.py'
PRECOMPUTE_WORKERS = 32
raw_root = Path(os.environ.get('OMNIXAS_DATA_ROOT', REPO_ROOT.parent / 'OmniXAS_data')) / 'materialscloud_omnixas_raw' / 'extracted'
print({
    'REPO_ROOT': str(REPO_ROOT),
    'SEED': SEED,
    'RUN_NAME': RUN_NAME,
    'OUTPUT_ROOT': str(OUTPUT_ROOT),
    'raw_root': str(raw_root),
    'raw_root_exists': raw_root.is_dir(),
})
assert PIPELINE.is_file(), f'Missing pipeline script: {PIPELINE}'


## Stage 1: graph-cache precompute + verify (one-time slow step)

Reads `cutoff` / `threebody_cutoff` from the `M3GNetXAS` encoder the pipeline
actually uses (`omnixas.model.m3gnet_xas`, defaults: cutoff 4.0, threebody_cutoff
4.0), derives the cache directory from them, then builds/refreshes the cache:

- every material in the train/val/test splits goes through the exact
  `CollateGraphs._build_graphs` code path, so the cache is identical to the
  on-the-fly build;
- one file per task/split, in canonical ID order; the header stores cutoff,
  threebody_cutoff, element_types, matgl/dgl/pymatgen/torch versions, and a hash
  of feff_graph.py;
- missing files are built (32 processes), valid files are skipped, stale files
  are deleted and rebuilt with a loud log (never silently reused);
- `verify=True` then builds ~20 sampled materials both ways and requires exact
  equality of graph topology, node/edge tensors, and encoder features.

Rerunning this cell is cheap: it only builds what is missing or stale.


In [ ]:
# import pgc first: it puts the repo root on sys.path for the omnixas import
import precompute_feff_graphs as pgc
from omnixas.model.m3gnet_xas import M3GNetXAS

# the pipeline's encoder settings, read from the model (not from flags)
encoder = M3GNetXAS().encoder
CUTOFF, THREEBODY_CUTOFF = encoder.cutoff, encoder.threebody_cutoff
GRAPH_CACHE = pgc.default_out_dir(REPO_ROOT, CUTOFF, THREEBODY_CUTOFF)
print(f'encoder settings: cutoff={CUTOFF} threebody_cutoff={THREEBODY_CUTOFF} '
      f'element_types={len(encoder.element_types)} feature_dim default=64')
print(f'graph cache: {GRAPH_CACHE}')

# one-time slow step (~15k materials); rerun is cheap (valid files are skipped)
pgc.ensure_graph_cache(REPO_ROOT, raw_root, GRAPH_CACHE, CUTOFF, THREEBODY_CUTOFF,
                       workers=PRECOMPUTE_WORKERS, rebuild_stale=True, verify=True)
print(f'graph cache ready and verified: {GRAPH_CACHE}')


## Stage 2: full pipeline run from the verified cache

Runs `train_m3gnet_xas_pipeline.py` with `--graph-cache` (no on-the-fly graph
building, `--num-workers 0`) and `--seed 45555`. The pipeline re-checks the
cache guard at load time; it never silently reuses a stale cache.

Resume: if the run is interrupted, set `RESUME = True` and rerun this cell. The
pipeline reuses the encoder checkpoint, exported features, and completed heads.


In [ ]:
RESUME = False  # set True to resume an interrupted run

cmd = [
    sys.executable, str(PIPELINE),
    '--run-name', RUN_NAME,
    '--output-root', str(OUTPUT_ROOT),
    '--seed', str(SEED),
    '--graph-cache', str(GRAPH_CACHE),
    '--num-workers', '0',
]
if RESUME:
    cmd.append('--resume')
subprocess.run(cmd, check=True)
print(f'pipeline complete: {OUTPUT_ROOT / RUN_NAME}')


## How to read the output / caveats

- Run artifacts under `output/training/m3gnet_xas_pipeline/m3gnet_xas_precomputation_showcase/`:
  `best_encoder.ckpt`, `features/{task}_{split}_{X,y}.txt`, `heads/universalXAS`,
  `heads/tunedUniversalXAS/{task}`, `RUN_COMPLETE.json`, and the CSV evaluations.
- Graph cache: `output/graph_cache/feff_graphs_c4_tb4/`. One directory per
  cutoff/threebody setting; this run's cache was verified before use and is
  re-guarded at load. Other settings (e.g. the r3 c5 set) keep their own
  directories and never mix.
- Selection: validation only; test is evaluated exactly once by the pipeline.
- `--encoder-rows-per-element` defaults to 12, tuned in the CPU-bound regime.
  After the cache switch the bottleneck is the GPU. Re-select on
  `val_balanced_rel_mse` before comparing results with on-the-fly runs.
- This is a standalone experiment (seed 45555, default 4.0/4.0 encoder). Do not
  mix its encoder/head artifacts with the r1/r2/r3 sweep artifacts.
